In [ ]:
!pip install fuzzywuzzy[speedup]


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 45.7 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
from fuzzywuzzy import fuzz
from fuzzywuzzy import process

In [ ]:
# Carregar o dataset
df = pd.read_excel(
    "KPMG_VI_New_raw_data_update_final.xlsx",
    sheet_name="Transactions",
    header=1
)


In [ ]:
print(df.head())

   transaction_id  product_id  customer_id transaction_date  online_order  \
0               1           2         2950       2017-02-25           0.0   
1               2           3         3120       2017-05-21           1.0   
2               3          37          402       2017-10-16           0.0   
3               4          88         3135       2017-08-31           0.0   
4               5          78          787       2017-10-01           1.0   

  order_status           brand product_line product_class product_size  \
0     Approved           Solex     Standard        medium       medium   
1     Approved   Trek Bicycles     Standard        medium        large   
2     Approved      OHM Cycles     Standard           low       medium   
3     Approved  Norco Bicycles     Standard        medium       medium   
4     Approved  Giant Bicycles     Standard        medium        large   

   list_price  standard_cost  product_first_sold_date  
0       71.49          53.62        

In [ ]:
# Verificar duplicatas exatas
duplicatas_exatas = df.duplicated().sum()
print(f"Número de duplicatas exatas: {duplicatas_exatas}")

Número de duplicatas exatas: 0


In [ ]:
# Remover duplicatas exatas
df_remover_duplicatas = df.drop_duplicates()

In [ ]:
print(df.columns)


Index(['transaction_id', 'product_id', 'customer_id', 'transaction_date',
       'online_order', 'order_status', 'brand', 'product_line',
       'product_class', 'product_size', 'list_price', 'standard_cost',
       'product_first_sold_date'],
      dtype='object')


In [ ]:
subset = [
    'customer_id',
    'transaction_date',
    'product_id',
    'brand',
    'product_line',
    'product_size'
]


In [ ]:
# Detectar duplicatas aproximadas
from fuzzywuzzy import fuzz

df2 = df_remover_duplicatas.copy()

duplicatas_aproximadas = []

for i in range(len(df2)): # percorrer o dataset
    for j in range(i + 1, len(df2)):
        score = fuzz.token_sort_ratio(
            str(df2.loc[i, subset].values),
            str(df2.loc[j, subset].values)
        )

        if score > 90:   # limiar
            duplicatas_aproximadas.append((i, j, score))


Não está com o resultado, pois o dataset tem muitos dados, e não consegui rodar no meu !!!!

In [ ]:
# Tratando duplicatas aproximadas
df2['transaction_date'] = pd.to_datetime(df2['transaction_date'])

remover = []

for (i, j, score) in duplicatas_aproximadas:
    linha1 = df2.loc[i]
    linha2 = df2.loc[j]

    # Manter o registro mais recente
    if linha1['transaction_date'] > linha2['transaction_date']:
        remover.append(j)
    else:
        remover.append(i)

df_final = df2.drop(remover)


In [ ]:
# Impacto das analises.
total_inicial = len(df)
total_pos_exatas = len(df_sem_dup_exatas)
total_final = len(df_final)

impacto = pd.DataFrame({
    "Descrição": [
        "Total inicial",
        "Duplicatas exatas removidas",
        "Duplicatas aproximadas removidas",
        "Total final"
    ],
    "Quantidade": [
        total_inicial,
        total_inicial - total_pos_exatas,
        total_pos_exatas - total_final,
        total_final
    ]
})

impacto
